<a href="https://colab.research.google.com/github/Maryam-Skaik/bike-sharing-demand-prediction/blob/main/notebook/bike_sharing_demand_random_forest.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Bike Sharing Regression

## Imports

In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer

from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer

from sklearn.ensemble import RandomForestRegressor
from sklearn import set_config
set_config(transform_output='pandas')

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, root_mean_squared_error

## Custom Evaluation Functions

In [ ]:
def regression_metrics(y_true, y_pred, label='', verbose = True, output_dict=False):
  # Get metrics
  mae = mean_absolute_error(y_true, y_pred)
  mse = mean_squared_error(y_true, y_pred)
  rmse = root_mean_squared_error(y_true, y_pred)
  r_squared = r2_score(y_true, y_pred)
  if verbose == True:
    # Print Result with Label and Header
    header = "-"*60
    print(header, f"Regression Metrics: {label}", header, sep='\n')
    print(f"- MAE = {mae:,.3f}")
    print(f"- MSE = {mse:,.3f}")
    print(f"- RMSE = {rmse:,.3f}")
    print(f"- R^2 = {r_squared:,.3f}")
  if output_dict == True:
      metrics = {'Label':label, 'MAE':mae, 'MSE':mse, 'RMSE':rmse, 'R^2':r_squared}
      return metrics

def evaluate_regression(reg, X_train, y_train, X_test, y_test, verbose = True, output_frame=False):
  # Get predictions for training data
  y_train_pred = reg.predict(X_train)

  # Call the helper function to obtain regression metrics for training data
  results_train = regression_metrics(y_train, y_train_pred, verbose = verbose, output_dict=output_frame, label='Training Data')
  print()

  # Get predictions for test data
  y_test_pred = reg.predict(X_test)

  # Call the helper function to obtain regression metrics for test data
  results_test = regression_metrics(y_test, y_test_pred, verbose = verbose, output_dict=output_frame, label='Test Data')

  # Store results in a dataframe if ouput_frame is True
  if output_frame:
    results_df = pd.DataFrame([results_train,results_test])

    # Set the label as the index
    results_df = results_df.set_index('Label')

    # Set index.name to none to get a cleaner looking result
    results_df.index.name=None

    # Return the dataframe
    return results_df.round(3)

## Load the Dataset


In [ ]:
fpath = '/content/drive/MyDrive/AXSOSACADEMY/02-MachineLearning/Practice/Data/day.csv'
df = pd.read_csv(fpath)
df.head()

,instant,dteday,season,yr,mnth,holiday,weekday,workingday,weathersit,temp,atemp,hum,windspeed,casual,registered,cnt
0,1,2011-01-01,1,0,1,0,6,0,2,0.344167,0.363625,0.805833,0.160446,331,654,985
1,2,2011-01-02,1,0,1,0,0,0,2,0.363478,0.353739,0.696087,0.248539,131,670,801
2,3,2011-01-03,1,0,1,0,1,1,1,0.196364,0.189405,0.437273,0.248309,120,1229,1349
3,4,2011-01-04,1,0,1,0,2,1,1,0.200000,0.212122,0.590435,0.160296,108,1454,1562
4,5,2011-01-05,1,0,1,0,3,1,1,0.226957,0.229270,0.436957,0.186900,82,1518,1600


### Target
- `cnt = casual + registered`
- Most rentals come from **registered users** → likely commuters

### Time & Calendar
- Rentals increase after weekends/holidays
- **Working days → higher demand**

### Weather Impact
- Better weather (`weathersit = 1`) → more rentals
- Worse weather → fewer rentals

### Temperature
- Slight increase in temp → more rentals
- Likely positive relationship

### Humidity & Wind
- High humidity → fewer rentals
- Wind may also reduce usage

### Trend
- Clear increase in rentals after holiday/weekend

### Key Features (for ML)
- Important:
  - `workingday`, `weathersit`, `temp`, `season`
- Also useful:
  - `humidity`, `windspeed`

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 731 entries, 0 to 730
Data columns (total 16 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   instant     731 non-null    int64  
 1   dteday      731 non-null    object 
 2   season      731 non-null    int64  
 3   yr          731 non-null    int64  
 4   mnth        731 non-null    int64  
 5   holiday     731 non-null    int64  
 6   weekday     731 non-null    int64  
 7   workingday  731 non-null    int64  
 8   weathersit  731 non-null    int64  
 9   temp        731 non-null    float64
 10  atemp       731 non-null    float64
 11  hum         731 non-null    float64
 12  windspeed   731 non-null    float64
 13  casual      731 non-null    int64  
 14  registered  731 non-null    int64  
 15  cnt         731 non-null    int64  
dtypes: float64(4), int64(11), object(1)
memory usage: 91.5+ KB


- Dataset contains 731 rows and 16 columns.

- No missing values, no need for imputation in preprocessing.

- All datatypes are consistent with featres

In [ ]:
df.describe().round(2)

,instant,season,yr,mnth,holiday,weekday,workingday,weathersit,temp,atemp,hum,windspeed,casual,registered,cnt
count,731.00,731.00,731.0,731.00,731.00,731.0,731.00,731.00,731.00,731.00,731.00,731.00,731.00,731.00,731.00
mean,366.00,2.50,0.5,6.52,0.03,3.0,0.68,1.40,0.50,0.47,0.63,0.19,848.18,3656.17,4504.35
std,211.17,1.11,0.5,3.45,0.17,2.0,0.47,0.54,0.18,0.16,0.14,0.08,686.62,1560.26,1937.21
min,1.00,1.00,0.0,1.00,0.00,0.0,0.00,1.00,0.06,0.08,0.00,0.02,2.00,20.00,22.00
25%,183.50,2.00,0.0,4.00,0.00,1.0,0.00,1.00,0.34,0.34,0.52,0.13,315.50,2497.00,3152.00
50%,366.00,3.00,1.0,7.00,0.00,3.0,1.00,1.00,0.50,0.49,0.63,0.18,713.00,3662.00,4548.00
75%,548.50,3.00,1.0,10.00,0.00,5.0,1.00,2.00,0.66,0.61,0.73,0.23,1096.00,4776.50,5956.00
max,731.00,4.00,1.0,12.00,1.00,6.0,1.00,3.00,0.86,0.84,0.97,0.51,3410.00,6946.00,8714.00


### Target (`cnt`)
- Mean ≈ **4504 rentals/day**
- Min = 22, Max = 8714 → **very large variation**
- High std (≈1937) → demand is **highly variable**

### User Types
- Registered mean ≈ **3656**
- Casual mean ≈ **848**

Most users are **registered (~80%)** → strong commuting pattern

### Working Days
- `workingday` mean ≈ **0.684**
  - ~68% of days are working days

### Holidays
- `holiday` mean ≈ **0.029**
  - Only ~3% of days are holidays

### Weather
- `weathersit` mean ≈ **1.39**
  - Most days have **good weather (1–2)**

### Temperature
- Mean ≈ **0.495** (moderate)
- Range: **0.059 → 0.862**

### Humidity
- Mean ≈ **0.628**
- Can reach up to **0.97**

### Wind Speed
- Mean ≈ **0.19**
- Max ≈ **0.50**

### Time Coverage
- `yr` mean ≈ **0.5**
  - Balanced between **2011 and 2012**
- `mnth` mean ≈ **6.5**
  - Covers **full year evenly**

In [ ]:
df.duplicated().sum()

np.int64(0)

In [ ]:
df.columns

Index(['instant', 'dteday', 'season', 'yr', 'mnth', 'holiday', 'weekday',
       'workingday', 'weathersit', 'temp', 'atemp', 'hum', 'windspeed',
       'casual', 'registered', 'cnt'],
      dtype='object')

In [ ]:
df = df.drop(columns= ['instant', 'dteday', 'casual', 'registered'])

## Split data

In [ ]:
y = df['cnt']
X = df.drop(columns='cnt')

X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)

## Preprocessing

In [ ]:
num_cols = X_train.select_dtypes('number').columns
scaler = StandardScaler()

preprocessor = ColumnTransformer(['num', scaler, num_cols], verbose_feature_names_out=False)
preprocessor

ColumnTransformer(transformers=['num', StandardScaler(),
                                Index(['season', 'yr', 'mnth', 'holiday', 'weekday', 'workingday',
       'weathersit', 'temp', 'atemp', 'hum', 'windspeed'],
      dtype='object')],
                  verbose_feature_names_out=False)

## Model Pipeline

In [ ]:
rf = RandomForestRegressor()
# rf_pipe = make_pipeline(preprocessor, rf)

In [ ]:
rf.fit(X_train, y_train)

RandomForestRegressor()

In [ ]:
evaluate_regression(rf, X_train, y_train, X_test, y_test)

------------------------------------------------------------
Regression Metrics: Training Data
------------------------------------------------------------
- MAE = 179.009
- MSE = 66,928.325
- RMSE = 258.705
- R^2 = 0.982

------------------------------------------------------------
Regression Metrics: Test Data
------------------------------------------------------------
- MAE = 415.207
- MSE = 408,899.709
- RMSE = 639.453
- R^2 = 0.894


In [ ]:
rf.get_params()

{'bootstrap': True,
 'ccp_alpha': 0.0,
 'criterion': 'squared_error',
 'max_depth': None,
 'max_features': 1.0,
 'max_leaf_nodes': None,
 'max_samples': None,
 'min_impurity_decrease': 0.0,
 'min_samples_leaf': 1,
 'min_samples_split': 2,
 'min_weight_fraction_leaf': 0.0,
 'monotonic_cst': None,
 'n_estimators': 100,
 'n_jobs': None,
 'oob_score': False,
 'random_state': None,
 'verbose': 0,
 'warm_start': False}

In [ ]:
rf_params = {
    'max_depth' : [10, 50, 150, 300],
    'min_samples_leaf' : [2, 3, 4],
    'n_estimators' : [100, 200, 300],
    'max_features': ['sqrt', 'log2'],
    'min_samples_split': [2, 5, 10]
}

In [ ]:
gs = GridSearchCV(rf, rf_params, n_jobs = -1, verbose = 1)

In [ ]:
gs.fit(X_train, y_train)

Fitting 5 folds for each of 216 candidates, totalling 1080 fits


GridSearchCV(estimator=RandomForestRegressor(), n_jobs=-1,
             param_grid={'max_depth': [10, 50, 150, 300],
                         'max_features': ['sqrt', 'log2'],
                         'min_samples_leaf': [2, 3, 4],
                         'min_samples_split': [2, 5, 10],
                         'n_estimators': [100, 200, 300]},
             verbose=1)

In [ ]:
gs.best_params_

{'max_depth': 300,
 'max_features': 'log2',
 'min_samples_leaf': 2,
 'min_samples_split': 5,
 'n_estimators': 300}

In [ ]:
evaluate_regression(gs.best_estimator_, X_train, y_train, X_test, y_test)

------------------------------------------------------------
Regression Metrics: Training Data
------------------------------------------------------------
- MAE = 289.818
- MSE = 168,692.680
- RMSE = 410.722
- R^2 = 0.954

------------------------------------------------------------
Regression Metrics: Test Data
------------------------------------------------------------
- MAE = 464.656
- MSE = 473,941.575
- RMSE = 688.434
- R^2 = 0.877


## Final Insights – Bike Sharing Demand Prediction (RandomForest)

### Objective
Predict daily bike rental demand (`cnt`) using environmental and calendar features.

### Model Used
- RandomForestRegressor
- Hyperparameter tuning applied using GridSearchCV

Best parameters:
- `n_estimators = 300`
- `max_depth = 300`
- `min_samples_leaf = 2`
- `min_samples_split = 5`
- `max_features = log2`

### Final Performance

#### Training Data
- MAE ≈ 289
- RMSE ≈ 411
- R² ≈ 0.954

#### Test Data
- MAE ≈ 465
- RMSE ≈ 688
- R² ≈ 0.877


### Key Observations

- The model performs well overall with strong predictive power (R² ≈ 0.88 on test data).
- Small gap between training and testing performance indicates **moderate overfitting**, but still acceptable generalization.
- Demand is strongly influenced by:
  - Weather conditions (`weathersit`, `temp`, `hum`)
  - Calendar features (`workingday`, `season`)
- Registered users dominate total demand, suggesting a commuter-driven pattern.

### Conclusion

A RandomForest model provides a solid baseline for bike demand prediction, achieving stable and reliable performance.